# AI Programming — Lecture 11
## Lab 2-3: ETTh1 Univariate Residual Forecasting

이 실습에서는 **MLP 구조는 그대로 유지하고 prediction target만 변경**합니다.

비교 대상:

1. Direct MLP
2. Last-Value Residual MLP
3. Seasonal Residual MLP
4. Last Value baseline
5. Seasonal Naive baseline

### 핵심 아이디어

예를 들어 마지막 관측값이 `5`라면 먼저

```text
baseline = [5, 5, 5, ..., 5]
```

를 만들고, MLP는 미래값 자체가 아니라 baseline에서의 변화량을 예측합니다.

```text
residual = [-0.2, 0.1, 0.4, ..., 1.3]
```

최종 예측:

```text
prediction = baseline + residual
```

### 학습 목표
- direct forecasting과 residual forecasting의 차이를 설명할 수 있습니다.
- last-value / seasonal anchor를 만들 수 있습니다.
- residual target을 구성할 수 있습니다.
- 동일 MLP에서 target representation만 바꾸었을 때 성능이 달라질 수 있음을 확인합니다.
- forecasting window alignment를 직접 검증할 수 있습니다.

### Colab 실행 안내
수업 시간 내 실습을 위해 기본 설정을 가볍게 조정했습니다.

- `BATCH_SIZE = 64`
- `MAX_EPOCHS = 150`
- 기본 seed는 하나만 사용: `[42]`
- early stopping 사용

추가 실험이 필요하면 seed를 `[42, 123, 2026]`으로 확장할 수 있습니다.

## 1. Forecasting Formulation

### Direct Forecasting

$$
\hat{\mathbf y}=f_\theta(\mathbf x)
$$

미래값 자체를 바로 예측합니다.

### Last-Value Residual Forecasting

마지막 관측값을 미래 구간의 기준으로 사용합니다.

$$
\mathbf b_{\text{last}}
=
[x_t,x_t,\ldots,x_t]
$$

모델 target은

$$
\mathbf r_{\text{last}}
=
\mathbf y-\mathbf b_{\text{last}}
$$

이고 최종 예측은

$$
\hat{\mathbf y}
=
\mathbf b_{\text{last}}+\hat{\mathbf r}_{\text{last}}
$$

입니다.

### Seasonal Residual Forecasting

시간 단위 ETTh1에서는 직전 24시간을 baseline으로 사용할 수 있습니다.

$$
\mathbf b_{\text{seasonal}}
=
[x_{t-23},\ldots,x_t]
$$

Residual output이 모두 0이면 최종 prediction은 각각의 naive baseline과 동일합니다.

## 2. 실습 환경 설정

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping

# ------------------------------------------------------------
# Common settings
# ------------------------------------------------------------
INPUT_LEN = 96
PRED_LEN = 24

BATCH_SIZE = 64
MAX_EPOCHS = 150
LEARNING_RATE = 1e-3
PATIENCE = 15

SEEDS = [42]
SHUFFLE = False
VERBOSE = 1

# Residual head를 0으로 초기화하면 학습 시작 시 anchor와 같은 예측을 합니다.
ZERO_INIT_RESIDUAL_HEAD = True

def reset_random_state(seed):
    tf.keras.backend.clear_session()
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

reset_random_state(SEEDS[0])

print("TensorFlow version:", tf.__version__)

# Colab runtime 확인
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus[0].name if gpus else '사용하지 않음 (CPU)')

## 3. ETTh1 데이터 불러오기

이번 실습도 `OT`만 사용합니다.

```text
MyDrive/Colab Notebooks/data/ETTh1.csv
```

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Google Colab이 아닙니다. FILE_PATH에 로컬 경로를 지정하세요.")

FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/ETTh1.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {FILE_PATH}\n"
        "FILE_PATH를 실제 ETTh1.csv 위치로 수정하세요."
    )

df = pd.read_csv(FILE_PATH)
df["date"] = pd.to_datetime(df["date"])

print("Data shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
display(df[["date", "OT"]].head())

plt.figure(figsize=(12, 4))
plt.plot(df["date"], df["OT"])
plt.xlabel("Date")
plt.ylabel("Oil Temperature (°C)")
plt.title("ETTh1: Oil Temperature")
plt.grid(True)
plt.show()

## 4. Chronological Split과 Scaling

```text
Train      70%
Validation 15%
Test       15%
```

Train 구간에서만 `StandardScaler`를 fitting합니다.

In [ ]:
ot_raw = df[["OT"]].to_numpy(dtype=np.float32)

n = len(ot_raw)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

ot_scaler = StandardScaler()
ot_scaler.fit(ot_raw[:train_end])

ot_scaled = ot_scaler.transform(ot_raw).astype(np.float32)

split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Start Index": [0, train_end, val_end],
    "End Index": [train_end, val_end, n],
    "Rows": [train_end, val_end - train_end, n - val_end],
})

display(split_summary)

## 5. Boundary-Aware Sliding Window

전체 시계열에서 window를 만든 뒤 **future target의 위치**를 기준으로 split을 결정합니다.

이렇게 하면 validation/test의 첫 target도 이전 split의 마지막 96시간을 정상적인 input context로 사용할 수 있습니다.

In [ ]:
def create_all_windows(series, input_len, pred_len):
    X, y, input_starts, target_starts, target_ends = [], [], [], [], []

    max_start = len(series) - input_len - pred_len + 1

    for input_start in range(max_start):
        target_start = input_start + input_len
        target_end = target_start + pred_len

        X.append(series[input_start:target_start])
        y.append(series[target_start:target_end, 0])

        input_starts.append(input_start)
        target_starts.append(target_start)
        target_ends.append(target_end)

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        np.asarray(input_starts),
        np.asarray(target_starts),
        np.asarray(target_ends),
    )


X_all, y_all, input_starts, target_starts, target_ends = create_all_windows(
    ot_scaled,
    INPUT_LEN,
    PRED_LEN,
)

train_mask = target_ends <= train_end
val_mask = (target_starts >= train_end) & (target_ends <= val_end)
test_mask = (target_starts >= val_end) & (target_ends <= n)

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

train_target_starts = target_starts[train_mask]
val_target_starts = target_starts[val_mask]
test_target_starts = target_starts[test_mask]

window_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "X Shape": [str(X_train.shape), str(X_val.shape), str(X_test.shape)],
    "y Shape": [str(y_train.shape), str(y_val.shape), str(y_test.shape)],
    "First Target Index": [
        int(train_target_starts[0]),
        int(val_target_starts[0]),
        int(test_target_starts[0]),
    ],
})

display(window_summary)

assert X_train.shape[1:] == (INPUT_LEN, 1)
assert y_train.shape[1] == PRED_LEN
assert val_target_starts[0] == train_end
assert test_target_starts[0] == val_end

## 6. Window Alignment 확인

시계열 forecasting에서는 한 step만 어긋나도 잘못된 실험이 됩니다.

첫 test sample에서

```text
마지막 input = raw[target_start - 1]
첫 target    = raw[target_start]
```

가 정확히 일치하는지 확인합니다.

In [ ]:
sample_idx = 0
target_start = int(test_target_starts[sample_idx])

last_input_scaled = X_test[sample_idx, -1, 0]
first_target_scaled = y_test[sample_idx, 0]

last_input_actual = ot_scaler.inverse_transform(
    [[last_input_scaled]]
)[0, 0]
first_target_actual = ot_scaler.inverse_transform(
    [[first_target_scaled]]
)[0, 0]

raw_last_input = ot_raw[target_start - 1, 0]
raw_first_target = ot_raw[target_start, 0]

alignment_check = pd.DataFrame({
    "Value": [
        "Last input OT",
        "First target OT",
    ],
    "Window value (°C)": [
        last_input_actual,
        first_target_actual,
    ],
    "Raw-series value (°C)": [
        raw_last_input,
        raw_first_target,
    ],
    "Raw index": [
        target_start - 1,
        target_start,
    ],
})

display(alignment_check.style.format({
    "Window value (°C)": "{:.4f}",
    "Raw-series value (°C)": "{:.4f}",
}))

assert np.isclose(last_input_actual, raw_last_input, atol=1e-5)
assert np.isclose(first_target_actual, raw_first_target, atol=1e-5)

print("Alignment check passed.")

## 7. Anchor와 Residual Target 만들기

- Last-value anchor
- Seasonal anchor
- Direct target
- Last-value residual target
- Seasonal residual target

을 동일한 train/validation/test window에 대해 생성합니다.

In [ ]:
def make_last_value_anchor(X):
    last_value = X[:, -1, 0:1]
    return np.repeat(last_value, PRED_LEN, axis=1)


def make_seasonal_anchor(X):
    return X[:, -PRED_LEN:, 0].copy()


anchors = {
    "train": {
        "last": make_last_value_anchor(X_train),
        "seasonal": make_seasonal_anchor(X_train),
    },
    "val": {
        "last": make_last_value_anchor(X_val),
        "seasonal": make_seasonal_anchor(X_val),
    },
    "test": {
        "last": make_last_value_anchor(X_test),
        "seasonal": make_seasonal_anchor(X_test),
    },
}

targets = {
    "Direct MLP": {
        "train": y_train,
        "val": y_val,
        "test": y_test,
        "anchor_train": np.zeros_like(y_train),
        "anchor_val": np.zeros_like(y_val),
        "anchor_test": np.zeros_like(y_test),
        "is_residual": False,
    },
    "Last-Value Residual MLP": {
        "train": y_train - anchors["train"]["last"],
        "val": y_val - anchors["val"]["last"],
        "test": y_test - anchors["test"]["last"],
        "anchor_train": anchors["train"]["last"],
        "anchor_val": anchors["val"]["last"],
        "anchor_test": anchors["test"]["last"],
        "is_residual": True,
    },
    "Seasonal Residual MLP": {
        "train": y_train - anchors["train"]["seasonal"],
        "val": y_val - anchors["val"]["seasonal"],
        "test": y_test - anchors["test"]["seasonal"],
        "anchor_train": anchors["train"]["seasonal"],
        "anchor_val": anchors["val"]["seasonal"],
        "anchor_test": anchors["test"]["seasonal"],
        "is_residual": True,
    },
}

target_stats = []

for name, values in targets.items():
    target_stats.append({
        "Target": name,
        "Mean": values["train"].mean(),
        "Std": values["train"].std(),
        "Mean Absolute Value": np.abs(values["train"]).mean(),
    })

display(
    pd.DataFrame(target_stats).style.format({
        "Mean": "{:.4f}",
        "Std": "{:.4f}",
        "Mean Absolute Value": "{:.4f}",
    })
)

## 8. Naive Baseline

Residual model에서 residual prediction을 0으로 두면 anchor 자체가 prediction이 됩니다.

따라서 residual model은 강한 naive baseline에서 **필요한 수정량만 학습**하는 것으로 볼 수 있습니다.

In [ ]:
def inverse_target(values_scaled):
    original_shape = values_scaled.shape

    return ot_scaler.inverse_transform(
        values_scaled.reshape(-1, 1)
    ).reshape(original_shape)


def evaluate_predictions(y_true_scaled, y_pred_scaled):
    y_true_actual = inverse_target(y_true_scaled)
    y_pred_actual = inverse_target(y_pred_scaled)

    return {
        "MSE (Normalized)": mean_squared_error(
            y_true_scaled.ravel(),
            y_pred_scaled.ravel(),
        ),
        "MAE (Normalized)": mean_absolute_error(
            y_true_scaled.ravel(),
            y_pred_scaled.ravel(),
        ),
        "MSE (°C²)": mean_squared_error(
            y_true_actual.ravel(),
            y_pred_actual.ravel(),
        ),
        "MAE (°C)": mean_absolute_error(
            y_true_actual.ravel(),
            y_pred_actual.ravel(),
        ),
    }


baseline_records = []

for name, pred in {
    "Last Value": anchors["test"]["last"],
    "Seasonal Naive (24h)": anchors["test"]["seasonal"],
}.items():
    row = {
        "Model": name,
        "Type": "Baseline",
        "Seed": "-",
    }
    row.update(evaluate_predictions(y_test, pred))
    baseline_records.append(row)

baseline_df = pd.DataFrame(baseline_records)

display(
    baseline_df.style.format({
        "MSE (Normalized)": "{:.4f}",
        "MAE (Normalized)": "{:.4f}",
        "MSE (°C²)": "{:.4f}",
        "MAE (°C)": "{:.4f}",
    })
)

## 9. 공통 MLP

세 MLP는 같은 구조를 사용합니다.

```text
96 → 128 → 64 → 24
```

Residual model의 마지막 layer를 0으로 초기화하면 학습 시작 시 prediction은 정확히 anchor baseline과 같습니다.

In [ ]:
def build_mlp(is_residual):
    output_initializer = (
        "zeros"
        if is_residual and ZERO_INIT_RESIDUAL_HEAD
        else "glorot_uniform"
    )

    model = Sequential([
        Input(shape=(INPUT_LEN, 1)),
        Flatten(),
        Dense(
            128,
            activation="relu",
            kernel_initializer="he_normal",
        ),
        Dense(
            64,
            activation="relu",
            kernel_initializer="he_normal",
        ),
        Dense(
            PRED_LEN,
            kernel_initializer=output_initializer,
            bias_initializer="zeros",
        ),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="mse",
        metrics=["mae"],
    )

    return model


def make_callbacks():
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            mode="min",
            verbose=1,
        )
    ]

## 10. Model Training

Colab 실습에서는 각 formulation을 seed 42에서 한 번씩 학습합니다.

추가 실험이 필요하면 `SEEDS`를 늘려 random initialization에 따른 변동도 비교할 수 있습니다.

In [ ]:
MODEL_DIR = Path("/content/etth1_univariate_residual_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

validation_records = []
model_paths = {}
histories = {}

for experiment_name, formulation in targets.items():
    for seed in SEEDS:
        print("\n" + "=" * 80)
        print(experiment_name, "| seed =", seed)

        reset_random_state(seed)

        model = build_mlp(
            is_residual=formulation["is_residual"]
        )

        history = model.fit(
            X_train,
            formulation["train"],
            validation_data=(
                X_val,
                formulation["val"],
            ),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            shuffle=SHUFFLE,
            callbacks=make_callbacks(),
            verbose=VERBOSE,
        )

        best_index = int(np.argmin(history.history["val_loss"]))

        record = {
            "Experiment": experiment_name,
            "Seed": seed,
            "Parameters": model.count_params(),
            "Best Epoch": best_index + 1,
            "Best Val MSE": float(
                history.history["val_loss"][best_index]
            ),
            "Best Val MAE (Normalized)": float(
                history.history["val_mae"][best_index]
            ),
            "Best Val MAE (°C)": float(
                history.history["val_mae"][best_index]
                * ot_scaler.scale_[0]
            ),
        }

        validation_records.append(record)
        histories[(experiment_name, seed)] = history.history

        safe_name = "".join(
            ch if ch.isalnum() else "_"
            for ch in experiment_name
        ).strip("_")

        model_path = MODEL_DIR / f"{safe_name}_seed{seed}.keras"
        model.save(model_path, overwrite=True)
        model_paths[(experiment_name, seed)] = str(model_path)

validation_df = pd.DataFrame(validation_records)

validation_summary = (
    validation_df
    .groupby("Experiment", as_index=False)
    .agg(
        Val_MSE_Mean=("Best Val MSE", "mean"),
        Val_MSE_Std=("Best Val MSE", "std"),
        Val_MAE_C_Mean=("Best Val MAE (°C)", "mean"),
        Val_MAE_C_Std=("Best Val MAE (°C)", "std"),
        Best_Epoch_Mean=("Best Epoch", "mean"),
    )
    .sort_values("Val_MSE_Mean")
    .reset_index(drop=True)
)

display(
    validation_summary.style.format({
        "Val_MSE_Mean": "{:.4f}",
        "Val_MSE_Std": "{:.4f}",
        "Val_MAE_C_Mean": "{:.4f}",
        "Val_MAE_C_Std": "{:.4f}",
        "Best_Epoch_Mean": "{:.1f}",
    }).highlight_min(
        subset=["Val_MSE_Mean", "Val_MAE_C_Mean"],
        axis=0,
    )
)

selected_experiment = validation_summary.iloc[0]["Experiment"]
print("Validation-selected formulation:", selected_experiment)

## 11. Test 성능 평가

In [ ]:
test_records = []
prediction_store = {}

for experiment_name, formulation in targets.items():
    for seed in SEEDS:
        model = tf.keras.models.load_model(
            model_paths[(experiment_name, seed)]
        )

        raw_output = model.predict(X_test, verbose=0)

        if formulation["is_residual"]:
            final_prediction = (
                formulation["anchor_test"] + raw_output
            )
        else:
            final_prediction = raw_output

        metrics = evaluate_predictions(
            y_test,
            final_prediction,
        )

        test_records.append({
            "Model": experiment_name,
            "Type": (
                "Residual MLP"
                if formulation["is_residual"]
                else "Direct MLP"
            ),
            "Seed": seed,
            **metrics,
        })

        prediction_store[(experiment_name, seed)] = (
            final_prediction
        )

test_df = pd.DataFrame(test_records)

test_summary = (
    test_df
    .groupby(["Model", "Type"], as_index=False)
    .agg(
        MSE_Normalized_Mean=("MSE (Normalized)", "mean"),
        MSE_Normalized_Std=("MSE (Normalized)", "std"),
        MAE_Normalized_Mean=("MAE (Normalized)", "mean"),
        MAE_Normalized_Std=("MAE (Normalized)", "std"),
        MSE_C2_Mean=("MSE (°C²)", "mean"),
        MSE_C2_Std=("MSE (°C²)", "std"),
        MAE_C_Mean=("MAE (°C)", "mean"),
        MAE_C_Std=("MAE (°C)", "std"),
    )
    .sort_values("MSE_Normalized_Mean")
    .reset_index(drop=True)
)

display(
    test_summary.style.format({
        "MSE_Normalized_Mean": "{:.4f}",
        "MSE_Normalized_Std": "{:.4f}",
        "MAE_Normalized_Mean": "{:.4f}",
        "MAE_Normalized_Std": "{:.4f}",
        "MSE_C2_Mean": "{:.4f}",
        "MSE_C2_Std": "{:.4f}",
        "MAE_C_Mean": "{:.4f}",
        "MAE_C_Std": "{:.4f}",
    }).highlight_min(
        subset=[
            "MSE_Normalized_Mean",
            "MAE_Normalized_Mean",
            "MSE_C2_Mean",
            "MAE_C_Mean",
        ],
        axis=0,
    )
)

## 12. Baseline을 포함한 최종 비교

In [ ]:
model_rows = []

for _, row in test_summary.iterrows():
    model_rows.append({
        "Model": row["Model"],
        "Type": row["Type"],
        "MSE (Normalized)": row["MSE_Normalized_Mean"],
        "MAE (Normalized)": row["MAE_Normalized_Mean"],
        "MSE (°C²)": row["MSE_C2_Mean"],
        "MAE (°C)": row["MAE_C_Mean"],
    })

final_comparison_df = pd.concat(
    [
        baseline_df[[
            "Model",
            "Type",
            "MSE (Normalized)",
            "MAE (Normalized)",
            "MSE (°C²)",
            "MAE (°C)",
        ]],
        pd.DataFrame(model_rows),
    ],
    ignore_index=True,
)

display(
    final_comparison_df.style.format({
        "MSE (Normalized)": "{:.4f}",
        "MAE (Normalized)": "{:.4f}",
        "MSE (°C²)": "{:.4f}",
        "MAE (°C)": "{:.4f}",
    }).highlight_min(
        subset=[
            "MSE (Normalized)",
            "MAE (Normalized)",
            "MSE (°C²)",
            "MAE (°C)",
        ],
        axis=0,
    )
)

## 13. Anchor 대비 성능 향상

In [ ]:
baseline_lookup = baseline_df.set_index("Model")

improvement_records = []

residual_anchor_pairs = {
    "Last-Value Residual MLP": "Last Value",
    "Seasonal Residual MLP": "Seasonal Naive (24h)",
}

for residual_model, baseline_name in residual_anchor_pairs.items():
    residual_row = test_summary[
        test_summary["Model"] == residual_model
    ].iloc[0]
    baseline_row = baseline_lookup.loc[baseline_name]

    improvement_records.append({
        "Residual Model": residual_model,
        "Anchor Baseline": baseline_name,
        "MSE Change (%)": (
            residual_row["MSE_Normalized_Mean"]
            / baseline_row["MSE (Normalized)"]
            - 1
        ) * 100,
        "MAE Change (%)": (
            residual_row["MAE_Normalized_Mean"]
            / baseline_row["MAE (Normalized)"]
            - 1
        ) * 100,
    })

improvement_df = pd.DataFrame(improvement_records)

display(
    improvement_df.style.format({
        "MSE Change (%)": "{:+.2f}%",
        "MAE Change (%)": "{:+.2f}%",
    })
)

## 14. Learning Curve

In [ ]:
for experiment_name in targets:
    seed_rows = validation_df[
        validation_df["Experiment"] == experiment_name
    ].sort_values("Best Val MSE")

    representative_seed = int(seed_rows.iloc[0]["Seed"])
    history = histories[(experiment_name, representative_seed)]

    plt.figure(figsize=(9, 4))
    plt.plot(history["loss"], label="Training MSE")
    plt.plot(history["val_loss"], label="Validation MSE")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title(
        f"{experiment_name} "
        f"(representative seed={representative_seed})"
    )
    plt.legend()
    plt.grid(True)
    plt.show()

## 15. Forecast 시각화

같은 test sample에 대해 direct / last-value residual / seasonal residual prediction을 비교합니다.

In [ ]:
sample_idx = 0

history_actual = inverse_target(
    X_test[sample_idx, :, 0]
)

future_actual = inverse_target(
    y_test[sample_idx]
)

history_steps = np.arange(-(INPUT_LEN - 1), 1)
future_steps = np.arange(1, PRED_LEN + 1)

# 각 모델에서 validation MSE가 가장 낮았던 seed를 대표로 사용합니다.
representative_predictions = {}

for experiment_name in targets:
    representative_row = (
        validation_df[
            validation_df["Experiment"] == experiment_name
        ]
        .sort_values("Best Val MSE")
        .iloc[0]
    )
    representative_seed = int(representative_row["Seed"])

    representative_predictions[experiment_name] = inverse_target(
        prediction_store[
            (experiment_name, representative_seed)
        ][sample_idx]
    )

plt.figure(figsize=(12, 6))
plt.plot(
    history_steps,
    history_actual,
    label="Historical OT",
)
plt.plot(
    future_steps,
    future_actual,
    marker="o",
    label="Actual Future",
)

for experiment_name, pred in representative_predictions.items():
    plt.plot(
        future_steps,
        pred,
        marker="o",
        label=experiment_name,
    )

plt.axvline(0, linestyle="--")
plt.xlabel("Time Step")
plt.ylabel("Oil Temperature (°C)")
plt.title("24-Hour Forecast: Direct vs. Residual Forecasting")
plt.legend()
plt.grid(True)
plt.show()

## 16. Horizon별 MAE

24시간 예측 중 어느 horizon에서 오차가 커지는지 확인합니다.

In [ ]:
y_test_actual = inverse_target(y_test)

plt.figure(figsize=(10, 5))

for experiment_name in targets:
    model_predictions = []

    for seed in SEEDS:
        model_predictions.append(
            inverse_target(
                prediction_store[(experiment_name, seed)]
            )
        )

    mean_prediction = np.mean(
        np.stack(model_predictions, axis=0),
        axis=0,
    )

    horizon_mae = np.mean(
        np.abs(y_test_actual - mean_prediction),
        axis=0,
    )

    plt.plot(
        np.arange(1, PRED_LEN + 1),
        horizon_mae,
        marker="o",
        label=experiment_name,
    )

plt.xlabel("Forecast Horizon (hour)")
plt.ylabel("MAE (°C)")
plt.title("Horizon-Wise Test MAE")
plt.legend()
plt.grid(True)
plt.show()

## 17. 결과와 Validation-Selected Model 저장

In [ ]:
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/models/"
    "etth1_univariate_residual_forecasting"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

validation_df.to_csv(
    OUTPUT_DIR / "validation_runs.csv",
    index=False,
)
validation_summary.to_csv(
    OUTPUT_DIR / "validation_summary.csv",
    index=False,
)
test_df.to_csv(
    OUTPUT_DIR / "test_runs.csv",
    index=False,
)
test_summary.to_csv(
    OUTPUT_DIR / "test_summary.csv",
    index=False,
)
final_comparison_df.to_csv(
    OUTPUT_DIR / "final_comparison.csv",
    index=False,
)
improvement_df.to_csv(
    OUTPUT_DIR / "anchor_improvement.csv",
    index=False,
)

selected_seed_row = (
    validation_df[
        validation_df["Experiment"] == selected_experiment
    ]
    .sort_values("Best Val MSE")
    .iloc[0]
)

selected_seed = int(selected_seed_row["Seed"])
selected_model = tf.keras.models.load_model(
    model_paths[(selected_experiment, selected_seed)]
)

selected_model_path = (
    OUTPUT_DIR / "validation_selected_model.keras"
)
selected_model.save(
    selected_model_path,
    overwrite=True,
)

print("Validation-selected formulation:", selected_experiment)
print("Representative seed:", selected_seed)
print("Saved model:", selected_model_path)
print("Saved results:", OUTPUT_DIR)

## 18. 결과 해석 체크리스트

1. Direct MLP가 Last Value baseline을 이깁니까?
2. Last-Value Residual MLP는 Direct MLP보다 좋아집니까?
3. Seasonal Residual MLP는 어떤 경우에 유리할까요?
4. Residual target의 평균 절댓값이 direct target보다 작습니까?
5. Zero-initialized residual head의 장점은 무엇입니까?
6. Forecast가 첫 미래 step부터 크게 어긋나는 현상이 residual formulation에서 줄었습니까?
7. 어떤 horizon에서 MAE가 가장 큽니까?

> **모델 구조를 바꾸지 않아도 prediction target의 표현 방식만으로 성능이 달라질 수 있습니다.**